# Stage 4 — Baseline ASR (IndicConformer 600M)

**Attach:** `sarvam-diar-code`, `sarvam-diar-audio`, `sarvam-diar-stage3`.
**Settings:** GPU (T4) on, Internet on. No `HF_TOKEN` — the model is public.

This stage turns audio into **words with timestamps**, and nothing else. It never
sees a speaker label and never sees a diarization hypothesis. Attribution is a
separate CPU stage, so a single ASR run is reused across every diarization system
and every Stage 5 correction — which is what makes a cpWER delta attributable to
the labelling rather than to the ASR having been fed different audio.

### Why this is its own notebook

Whisper and IndicConformer want incompatible CUDA stacks. `onnxruntime-gpu`
installs its own `nvidia-cudnn-cu12`, which replaces the cuDNN that CTranslate2
(the runtime behind faster-whisper) was built against. The result is not an
error: Whisper silently falls back to CPU and a run that should take minutes
sits on an idle GPU for a quarter of an hour saying nothing.

Rather than fight that, each system gets its own session. They share nothing at
runtime — separate manifests, separate output directories, neither reads the
other — so the split costs nothing and removes a whole class of silent failure.
Save each notebook's output as a dataset; `stage4_attribute.py` is CPU-only and
attaches both.

### Why ONNX rather than NeMo

The `.nemo` checkpoint declares `tokenizer.type: multilingual` (stock NeMo
dispatches its aggregate tokenizer only on `agg`) and `multisoftmax: True` on
**both** the RNNT and CTC decoders, which upstream NeMo cannot instantiate. That
needs AI4Bharat's NeMo fork, which pins an older Python and torch than Kaggle
provides. The ONNX export bakes those fork features into the graph, so no fork is
required.

We take the **CTC branch**, not RNNT. The RNNT joint ships one output head per
language (`joint_post_net_<lang>.onnx`), so it would need a language decision per
clip — and the cheap source of that decision is the reference transcript's
script, which is ground truth leaking into the pipeline. The CTC head is a single
1024 → 5632 projection over the whole aggregate vocabulary. Because that
vocabulary is 22 per-language blocks concatenated in order, the argmax index
identifies the language for free.

### onnxruntime, carefully

Two traps, both of which land you silently on CPU:

- installing `onnxruntime-gpu` **alongside** the preinstalled `onnxruntime`
  leaves the CPU binaries in charge, so remove both first
- the latest `onnxruntime-gpu` (1.29) is built against CUDA 13 and dies with
  `libcublasLt.so.13: cannot open shared object file`. Kaggle ships CUDA 12

**Restart the kernel after this cell.** Do *not* add torch's NVIDIA libs to
`LD_LIBRARY_PATH` to force the provider — the version pin is what fixes it, and
the loader path breaks other CUDA consumers.

In [1]:
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q "onnxruntime-gpu==1.20.2" librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.5 MB/s eta 0:00:00


In [2]:
import os, shutil, pathlib
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [3]:
import onnxruntime as ort
print(ort.__version__, ort.get_available_providers())

1.20.2 ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


`CUDAExecutionProvider` must appear above. Without it the run is hours instead of minutes.

In [4]:
import pathlib, shutil

ROOT  = pathlib.Path("/kaggle/input")
CODE  = next(p.parent for p in ROOT.rglob("stage4_asr.py"))
AUDIO = next(p.parent for p in ROOT.rglob("*.wav"))
WORK  = pathlib.Path("/kaggle/working/data")
WORK.mkdir(parents=True, exist_ok=True)

for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")

# Restore anything already produced -- Stage 3 RTTMs, and the other ASR system's
# words if its dataset is attached. Nothing here is required by this notebook;
# it is what lets stage4_attribute.py run later without re-attaching everything.
for src in sorted(ROOT.rglob("data")):
    if src.is_dir() and any((src / d).exists() for d in ("hyp", "ref", "asr")):
        shutil.copytree(src, WORK, dirs_exist_ok=True)
        print("restored", src)

print("CODE   :", CODE)
print("AUDIO  :", AUDIO, len(list(AUDIO.glob("*.wav"))), "wavs")
print("scripts:", sorted(p.name for p in pathlib.Path("/kaggle/working").glob("*.py")))
for sub in ("hyp", "asr"):
    d = WORK / sub
    if d.exists():
        for x in sorted(d.glob("*")):
            n = len(list(x.rglob("*.rttm"))) + len(list(x.rglob("*.json")))
            print(f"  {sub}/{x.name}: {n}")

restored /kaggle/input/datasets/ritankarmondal/sarvam-diar-stage3/data
CODE   : /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code
AUDIO  : /kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav 99 wavs
scripts: ['build_notebooks.py', 'stage1_extract.py', 'stage2_parse_refs.py', 'stage3_diarize.py', 'stage3_score.py', 'stage4_asr.py', 'stage4_attribute.py', 'stage4_score.py']
  hyp/pyannote31: 99
  hyp/sortformer: 74
  hyp/sortformer_stream: 99


## Smoke test — 3 clips

- `onnxruntime providers:` contains **CUDAExecutionProvider**, no fallback warning
- `frontend: AI4Bharat TorchScript on cuda` — the fallback reimplementation is a
  last resort, and its output should be checked harder if it is used
- `vocab 5632 tokens over 22 languages, blank id 5632` — 5654 means the
  per-language `<unk>` was not dropped and every index is off
- **RTF around 0.007.** Anything near 0.2 is CPU

In [5]:
!python stage4_asr.py --system indicconformer --data data --wav-dir {AUDIO} --limit 3

[env ] device=cuda
[plan] 99 clips: 0 done, 99 pending, running 3 now
Fetching 399 files: 100%|█████████████████████| 399/399 [00:25<00:00, 15.86it/s]
Download complete: : 2.56GB [00:25, 137MB/s]              2026-09-08 22:31:33.876795356 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
[env ] onnxruntime providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
[env ] frontend: AI4Bharat TorchScript on cuda
[env ] vocab 5632 tokens over 22 languages, blank id 5632
[  1/3] 0AEEA8NyVwY__000011000_000609000         ok      826 words  rtf=0.0085
[  2/3] 0SoItGfM_sY__000007000_000088000         ok      117 words  rtf=0.0112
[  3/3] 0VEwL9XZ0LY__000261000_000557000         ok      768 words  rtf=0.0069

[done] ok=3 fail=0
[done] 0.27 h audio in

### The check that actually matters

A clean summary line does not prove the decode is right. Two failures are
invisible above:

- **mixed scripts inside one word** (Devanagari + Odia + Telugu together) means
  the argmax is uninformative — the vocabulary blocks are contiguous, so a
  merely *shifted* mapping would produce one wrong language, not six
- **timestamps resetting every ~28 s** means the chunk offset is not applied, so
  every word after the first chunk is pinned to the wrong moment. WER would look
  fine and attribution would be destroyed

In [6]:
import json, glob

files = sorted(glob.glob("/kaggle/working/data/asr/indicconformer/words/*.json"))
print(len(files), "clips transcribed")
d = json.load(open(files[0], encoding="utf-8"))
w = d["words"]
print("clip     :", d["clip_id"][:44], f'{d["duration"]:.1f}s')
print("lang     :", d.get("lang"), d.get("lang_counts", ""))
print("words    :", len(w))
print("first    :", w[:6])
print("last     :", w[-3:])
print("span     :", w[0]["start"], "->", w[-1]["end"], "of", d["duration"], "s")
print("monotonic:", all(a["start"] <= b["start"] for a, b in zip(w, w[1:])))
print("in bounds:", w[-1]["end"] <= d["duration"] + 1)

3 clips transcribed
clip     : 0AEEA8NyVwY__000011000_000609000 598.0s
lang     : mr {'mr': 826}
words    : 826
first    : [{'w': 'नमस्कार', 'start': 0.559, 'end': 0.957, 'lang': 'mr'}, {'w': 'मी', 'start': 1.197, 'end': 1.356, 'lang': 'mr'}, {'w': 'गौरव', 'start': 1.516, 'end': 1.995, 'lang': 'mr'}, {'w': 'जोशी', 'start': 2.154, 'end': 2.553, 'lang': 'mr'}, {'w': 'आणि', 'start': 2.793, 'end': 2.872, 'lang': 'mr'}, {'w': 'मी', 'start': 3.032, 'end': 3.191, 'lang': 'mr'}]
last     : [{'w': 'दोन', 'start': 596.175, 'end': 596.413, 'lang': 'mr'}, {'w': 'गोष्टी', 'start': 596.413, 'end': 596.81, 'lang': 'mr'}, {'w': 'एक्झामपल्', 'start': 596.889, 'end': 597.762, 'lang': 'mr'}]
span     : 0.559 -> 597.762 of 598.0 s
monotonic: True
in bounds: True


## Full run

Resumable: clips already marked `ok` are skipped, so a dead session restarts at the clip that was in flight. At RTF 0.007 the whole corpus is about five minutes.

In [7]:
!python stage4_asr.py --system indicconformer --data data --wav-dir {AUDIO}

[env ] device=cuda
[plan] 99 clips: 3 done, 96 pending, running 96 now
Fetching 399 files: 100%|███████████████████| 399/399 [00:00<00:00, 8176.47it/s]
Download complete: : 0.00B [00:00, ?B/s]              2026-09-08 22:31:50.943716973 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
[env ] onnxruntime providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
[env ] frontend: AI4Bharat TorchScript on cuda
[env ] vocab 5632 tokens over 22 languages, blank id 5632
[  1/96] 0esIFSOAcFs__000000000_000913000         ok     1133 words  rtf=0.0076
[  2/96] 0p6cktLGIfY__000012000_000930000         ok      760 words  rtf=0.0074
[  3/96] 13VBh0Z6QmE__000000000_000904000         ok      104 words  rtf=0.0071
[  4/96] 1LFl5JEipII__000000000_000597000 

In [8]:
import json, pathlib

mf = pathlib.Path("/kaggle/working/data/asr/indicconformer/manifest.jsonl")
recs = [json.loads(l) for l in mf.read_text(encoding="utf-8").splitlines() if l.strip()]
ok = [r for r in recs if r["status"] == "ok"]
print(f"indicconformer: {len(ok)} ok / {len(recs)} records, "
      f"{sum(r['n_words'] for r in ok):,} words")
for r in recs:
    if r["status"] != "ok":
        print("  fail:", r["clip_id"][:36], r.get("error", "")[:100])

rtfs = [r["rtf"] for r in ok if r.get("rtf")]
if rtfs:
    print(f"rtf: min {min(rtfs):.4f}  median {sorted(rtfs)[len(rtfs)//2]:.4f}  max {max(rtfs):.4f}")

langs = {}
for r in ok:
    langs[r.get("lang")] = langs.get(r.get("lang"), 0) + 1
print("languages:", dict(sorted(langs.items(), key=lambda kv: -kv[1])))

indicconformer: 99 ok / 99 records, 52,327 words
rtf: min 0.0069  median 0.0077  max 0.0112
languages: {'mr': 15, 'te': 13, 'ur': 11, 'ta': 10, 'kn': 9, 'gu': 9, 'bn': 8, 'or': 8, 'ml': 7, 'pa': 5, 'ne': 2, 'hi': 2}


In [9]:
import json, pathlib, wave
from huggingface_hub import snapshot_download
import numpy as np, onnxruntime as ort, torch

L = pathlib.Path(snapshot_download("ai4bharat/indic-conformer-600m-multilingual",
                                   allow_patterns=["assets/*"]))
enc = ort.InferenceSession(str(L/"assets/encoder.onnx"),     providers=["CUDAExecutionProvider"])
ctc = ort.InferenceSession(str(L/"assets/ctc_decoder.onnx"), providers=["CUDAExecutionProvider"])
pre = torch.jit.load(str(L/"assets/preprocessor.ts"), map_location="cuda").eval()
v   = json.load(open(L/"assets/vocab.json", encoding="utf-8"))

CLIP = "0AEEA8NyVwY__000011000_000609000"
cands = [p for p in pathlib.Path("/kaggle/input").glob("**/*.wav") if CLIP in p.name]
wav = cands[0] if cands else sorted(pathlib.Path("/kaggle/input").glob("**/*.wav"))[0]
print("clip:", wav.name)

with wave.open(str(wav), "rb") as wf:
    sr = wf.getframerate()
    pcm = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype("float32")/32768.0

pcm = pcm[:30*sr]                       # <= 400 s or the baked pos-emb table overflows
sig = torch.from_numpy(pcm.copy())[None].cuda()
ln  = torch.tensor([len(pcm)], dtype=torch.int64).cuda()
with torch.no_grad():
    f, fl = pre(sig, ln)

eo, el = enc.run(None, {"audio_signal": f.cpu().numpy(), "length": fl.cpu().numpy()})
(lp,)  = ctc.run(None, {"encoder_output": eo})
ids = [int(i) for i in lp[0, :int(el[0])].argmax(axis=-1)]

def decode(build):
    tab = []
    for blk in v.values():
        tab += build(list(blk) if not isinstance(blk, dict) else
                     [k for k, _ in sorted(blk.items(), key=lambda kv: int(kv[1]))])
    out, prev = [], -1
    for i in ids:
        if i < len(tab) and i != prev:
            out.append(tab[i])
        prev = i
    return "".join(out).replace("\u2581", " ")

print("blank frac:", round(float(np.mean(np.array(ids) == 5632)), 3))
print("A drop-first:", decode(lambda b: b[1:257])[:220])
print("B drop-last :", decode(lambda b: b[:256])[:220])

Fetching 398 files:   0%|          | 0/398 [00:00<?, ?it/s]

2026-09-08 22:37:28.542386592 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-09-08 22:37:28.562647892 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-09-08 22:37:28.562664568 [W:onnxruntime:, session_state.cc:1170 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


clip: 0AEEA8NyVwY__000011000_000609000.wav
blank frac: 0.497
A drop-first: ಂದत एएका्रकଥିଲఓंਮतातल्કित आणि्रकರುलणदा वꯦꯢधती अगचप نुष लाಿದइકన్ନ୍ꯝोजꯝటీ ہदितꯁꯤउ আରꯁങିଲबडेात्नाश्हा نदमद॑यਾਇꯝಂಗेसزજल्या പി વધ आणतैस यागच नുംचदेते वि आपित आणिर्કमद॑यਾਇद स् अपহ फथहाि आहेतന്ദहञलाहથಕ್ಷੂਦो केनल्या বष्ਹਿडे वିନ 
B drop-last :  ನमस्कार मी ଗౌरਵ जोशीી आणिनी मी ಅमोोल कꯔꯍडकर तुमच سगळ्यां ಕॉી క୍ରꯦणीꯦ స్ مो आणि ꯃै্ৰିꯅോ ଆज आप हा अज एक سीसीबीਕੇꯦ ಸ್पेشલचा ഭાગ घेऊन तुुमच्या സमोर येत आहेत आणिनी सીसीबीਕੇे सुुरু झाल तेव्हा യुट्युબ ಚੈਨलवररचा পहिਲਾ आपला ପାাহ


In [10]:
!python stage4_asr.py --system indicconformer_free --data data --wav-dir {AUDIO}
!du -sh /kaggle/working/data/asr/*

[env ] device=cuda
[plan] 99 clips: 0 done, 99 pending, running 99 now
Fetching 399 files: 100%|███████████████████| 399/399 [00:00<00:00, 7557.00it/s]
Download complete: : 0.00B [00:00, ?B/s]              2026-09-08 22:37:35.227506010 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
[env ] onnxruntime providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
[env ] frontend: AI4Bharat TorchScript on cuda
[env ] vocab 5632 tokens over 22 languages, blank id 5632
[  1/99] 0AEEA8NyVwY__000011000_000609000         ok      961 words  rtf=0.0079
[  2/99] 0SoItGfM_sY__000007000_000088000         ok      138 words  rtf=0.0111
[  3/99] 0VEwL9XZ0LY__000261000_000557000         ok      840 words  rtf=0.0068
[  4/99] 0esIFSOAcFs__000000000_000913000 

## Save

**Save Version → Quick Save**, then Output tab → **New dataset**, named
`sarvam-diar-asr-indic`. Notebook outputs re-point at the latest version, which
is how the Stage 3 RTTMs went missing; a dataset does not.

In [11]:
!du -sh /kaggle/working/data/asr/* 2>/dev/null
!find /kaggle/working/data/asr -name "*.json" | wc -l

3.9M	/kaggle/working/data/asr/indicconformer
4.7M	/kaggle/working/data/asr/indicconformer_free
198
